In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from tensorflow.keras.datasets import imdb

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


cuda


In [17]:
VOCAB_SIZE = 10000
MAX_LEN = 200

(x_train,y_train),(x_test,y_test)=imdb.load_data(num_words=VOCAB_SIZE)

def pad(x):
    return np.array([i[:MAX_LEN]+[0]*(MAX_LEN-len(i)) for i in x])

x_train=pad(x_train)
x_test=pad(x_test)

x_train=torch.tensor(x_train)
y_train=torch.tensor(y_train)

x_test=torch.tensor(x_test)
y_test=torch.tensor(y_test)

print(x_train.shape)

torch.Size([25000, 200])


In [18]:
class RNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(VOCAB_SIZE,128)
        self.rnn = nn.GRU(128,128,batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(128,1)

    def forward(self,x):
        x = self.emb(x)
        out,_ = self.rnn(x)
        out = self.dropout(out[:,-1])
        return torch.sigmoid(self.fc(out))

In [21]:
model = RNNModel().to(device)

loss_fn = nn.BCELoss()
opt = optim.Adam(model.parameters(),0.001)

for epoch in range(5):
    perm = torch.randperm(len(x_train))
    correct = 0

    for i in range(0,10000,64):   # small subset for speed
        idx = perm[i:i+64]

        xb = x_train[idx].to(device)
        yb = y_train[idx].float().unsqueeze(1).to(device)

        opt.zero_grad()
        out = model(xb)
        loss = loss_fn(out,yb)
        loss.backward()
        opt.step()

        pred = (out>0.5).int()
        correct += (pred==yb.int()).sum().item()

    print("Epoch",epoch+1,"Accuracy:",correct/10000)

Epoch 1 Accuracy: 0.5107
Epoch 2 Accuracy: 0.5267
Epoch 3 Accuracy: 0.5293
Epoch 4 Accuracy: 0.5511
Epoch 5 Accuracy: 0.6699


In [22]:
with torch.no_grad():
    out = model(x_test[:2000].to(device))
    pred = (out>0.5).int().squeeze()
    acc = (pred.cpu()==y_test[:2000]).sum()/2000
    print("Test Accuracy:",acc.item())

Test Accuracy: 0.6625000238418579


In [23]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

def predict(text):
    word_index = imdb.get_word_index()
    seq = [word_index.get(w,0) for w in text.lower().split()]
    seq = pad_sequences([seq],MAX_LEN)

    t = torch.tensor(seq).to(device)
    out = model(t).item()

    print("Positive" if out>0.5 else "Negative")

predict("this movie was amazing and wonderful")
predict("worst movie ever")

Negative
Negative
